# Ноутбук 02b: Валидация метрик качества ответов

**Цель:** выбрать метрику для главного эксперимента (ноутбук 03), убедившись что она измеряет именно то, что нужно.

## Что делает этот ноутбук

1. Берёт 10 вопросов из нашего зафиксированного набора.
2. Генерирует ответы через baseline RAG на **чистой базе** (`rag_clean`).
3. Считает **4 метрики** для каждого ответа:
   - **Exact Match (EM)** — точное совпадение с эталоном (после нормализации).
   - **Token F1** — частичное совпадение по токенам (классика QA-бенчмарков).
   - **RAGAS** — faithfulness (верен ли ответ контексту) + answer_relevance (отвечает ли на вопрос).
   - **LLM-judge** — два судьи (`llama-3.3-70b` и `llama-3.1-8b`) оценивают правильность по шкале 0/1 с объяснением.
4. Выводит **таблицу** с результатами всех метрик рядом для ручной валидации.
5. Считает **inter-judge agreement** (Cohen's Kappa) между двумя LLM-судьями.
6. Считает **корреляцию** каждой метрики с вашей ручной оценкой (вы сами проставляете 0/1 для 10 вопросов).

## Почему это важно

EM/F1 часто занижают результат: если эталон «Ford Field in Detroit», а модель ответила «Ford Field» — EM=0, хотя ответ фактически верный. LLM-judge может это учесть. Но LLM-judge дорог и непрозрачен. RAGAS измеряет не «правильность», а «верность контексту» — другая ось. Сравнив всё на 10 примерах с вашей ручной оценкой, вы **эмпирически** выберете метрику для основного эксперимента — и это будет обосновано в дипломе.

## Шаг 0. Установка и инициализация

In [2]:
!pip install -q qdrant-client fastembed groq ragas langchain langchain-groq datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import random
import string
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ---- КОНФИГ ----
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

assert QDRANT_URL, "Не найден QDRANT_URL"
assert QDRANT_API_KEY, "Не найден QDRANT_API_KEY"
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, "Не найден GROQ_API_KEY"

SEED = 42
N_EVAL = 10  # вопросов для ручной валидации

COLLECTION_CLEAN  = "rag_clean"
EMBEDDING_MODEL   = "BAAI/bge-small-en-v1.5"
EMBEDDING_DIM     = 384

JUDGE_MODEL_STRONG = "llama-3.3-70b-versatile"
JUDGE_MODEL_FAST   = "llama-3.1-8b-instant"
GENERATOR_MODEL    = "llama-3.3-70b-versatile"  # тот же что и в основном эксперименте

ARTIFACTS_DIR = Path("/content/drive/MyDrive/rag_experiment/artifacts")
EVAL_DIR      = ARTIFACTS_DIR / "eval_metrics"
EVAL_DIR.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

print(f"Artifacts: {ARTIFACTS_DIR}")
print(f"Eval output: {EVAL_DIR}")

Artifacts: /content/drive/MyDrive/rag_experiment/artifacts
Eval output: /content/drive/MyDrive/rag_experiment/artifacts/eval_metrics


In [5]:
# Загружаем артефакты
with open(ARTIFACTS_DIR / "questions.json", encoding="utf-8") as f:
    all_questions = json.load(f)
with open(ARTIFACTS_DIR / "gold_mapping.json", encoding="utf-8") as f:
    gold_mapping = json.load(f)

# Берём первые N_EVAL вопросов (детерминировано, seed зафиксирован)
rng = random.Random(SEED)
eval_questions = rng.sample(all_questions, N_EVAL)

print(f"Вопросов для валидации: {len(eval_questions)}")
print(f"Типы: {Counter(q['type'] for q in eval_questions)}")
print(f"Уровни: {Counter(q['level'] for q in eval_questions)}")
print("\nВопросы:")
for i, q in enumerate(eval_questions):
    print(f"  {i+1}. [{q['type']}] {q['question']}")
    print(f"      → {q['answer']}")

Вопросов для валидации: 10
Типы: Counter({'bridge': 7, 'comparison': 3})
Уровни: Counter({'hard': 10})

Вопросы:
  1. [comparison] Are both Jack and Coke and Clover Club Cocktail cocktails?
      → yes
  2. [bridge] What is the uppermost age range for the sort of fiction written by Alexander Gordon Smith?
      → early 20s
  3. [bridge] What other method of extending an ice hockey game exists other than the one used in the 1932 Allan Cup?
      → the shootout
  4. [comparison] Between Danny Elfman and Fran Healy who has worked in more diverse fields?
      → Elfman
  5. [bridge] What Japanese fashion label was founded by the creator of Dover Street Market?
      → Comme des Garçons
  6. [bridge] Which of Tara Strong major voice role in animated series is an American animated television series based on the DC Comics fictional superhero team, the "Teen Titans"?
      → Teen Titans Go!
  7. [bridge] This aircraft carrier served as a recovery ship for this flight which circled the earth ho

## Шаг 1. Подключение к сервисам

In [6]:
import time
from groq import Groq
from qdrant_client import QdrantClient
from fastembed import TextEmbedding

qdrant   = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60)
groq_cli = Groq(api_key=GROQ_API_KEY)
embedder = TextEmbedding(model_name=EMBEDDING_MODEL)

info = qdrant.get_collection(COLLECTION_CLEAN)
print(f"rag_clean: {info.points_count} точек")

def embed(texts):
    return [v.tolist() for v in embedder.embed(texts)]

def groq_complete(prompt, model, max_tokens=512, temperature=0.0):
    """Простой вызов Groq без rate-limiter (10 вопросов — не критично)."""
    time.sleep(0.3)
    resp = groq_cli.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content.strip()

print("Все сервисы подключены.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

rag_clean: 7997 точек
Все сервисы подключены.


## Шаг 2. Baseline RAG — получение ответов

Простейший пайплайн: dense retrieval top-5 из `rag_clean` → prompt → ответ.
Используем тот же генератор, что планируется для основного эксперимента.

In [7]:
RAG_PROMPT = """Answer the question based ONLY on the provided context.
Be concise — give the answer in 1-3 words when possible.
If the context does not contain enough information, respond with: "I don't know".

Context:
{context}

Question: {question}

Answer:"""

def retrieve(question, top_k=5):
    qvec = embed([question])[0]
    hits = qdrant.query_points(
        collection_name=COLLECTION_CLEAN,
        query=qvec,
        limit=top_k,
        with_payload=True,
    ).points
    return hits

def rag_answer(question, top_k=5):
    hits = retrieve(question, top_k=top_k)
    context_parts = []
    for i, h in enumerate(hits):
        title = h.payload.get("title", "Unknown")
        text  = h.payload.get("text", "")
        context_parts.append(f"[{i+1}] {title}\n{text}")
    context = "\n\n".join(context_parts)
    prompt  = RAG_PROMPT.format(context=context, question=question)
    answer  = groq_complete(prompt, model=GENERATOR_MODEL, max_tokens=100)
    return answer, hits, context

# Генерируем ответы для всех 10 вопросов
print("Генерирую ответы...")
results = []
for q in tqdm(eval_questions):
    pred, hits, context = rag_answer(q["question"])
    results.append({
        "id":       q["id"],
        "question": q["question"],
        "gold":     q["answer"],
        "type":     q["type"],
        "predicted": pred,
        "context":  context,
        "retrieved_titles": [h.payload.get("title") for h in hits],
    })

# Быстрый просмотр
print("\nОтветы:")
for r in results:
    print(f"  Q: {r['question'][:70]}")
    print(f"     Gold:      {r['gold']}")
    print(f"     Predicted: {r['predicted']}")
    print()

Генерирую ответы...


  0%|          | 0/10 [00:00<?, ?it/s]


Ответы:
  Q: Are both Jack and Coke and Clover Club Cocktail cocktails?
     Gold:      yes
     Predicted: Yes

  Q: What is the uppermost age range for the sort of fiction written by Ale
     Gold:      early 20s
     Predicted: Early 20s

  Q: What other method of extending an ice hockey game exists other than th
     Gold:      the shootout
     Predicted: Shootout

  Q: Between Danny Elfman and Fran Healy who has worked in more diverse fie
     Gold:      Elfman
     Predicted: Danny Elfman

  Q: What Japanese fashion label was founded by the creator of Dover Street
     Gold:      Comme des Garçons
     Predicted: Comme des Garçons

  Q: Which of Tara Strong major voice role in animated series is an America
     Gold:      Teen Titans Go!
     Predicted: Teen Titans Go!

  Q: This aircraft carrier served as a recovery ship for this flight which 
     Gold:      three
     Predicted: John Glenn's, 3 times

  Q: In what Country is Sul America Esporte Clube in?
     Gold:      Braz

## Шаг 3. Метрика 1 — Exact Match и Token F1

Классические метрики из SQuAD/NQ. Нормализация: lowercase, удаление артиклей и пунктуации.

**EM** = 1 если нормализованные строки совпадают точно.

**F1** = гармоническое среднее precision и recall по токенам. Позволяет засчитать частичный ответ.

In [8]:
def normalize_answer(s):
    """Lowercase, удаление артиклей, пунктуации и лишних пробелов."""
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    s = ' '.join(s.split())
    return s

def token_f1(pred, gold):
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    common = Counter(pred_tokens) & Counter(gold_tokens)
    n_common = sum(common.values())
    if n_common == 0:
        return 0.0
    precision = n_common / len(pred_tokens)
    recall    = n_common / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def exact_match(pred, gold):
    return int(normalize_answer(pred) == normalize_answer(gold))

for r in results:
    r["em"] = exact_match(r["predicted"], r["gold"])
    r["f1"] = token_f1(r["predicted"],  r["gold"])

print(f"Среднее EM: {sum(r['em'] for r in results) / len(results):.2f}")
print(f"Среднее F1: {sum(r['f1'] for r in results) / len(results):.2f}")
print("\nПо вопросам:")
for r in results:
    print(f"  EM={r['em']} F1={r['f1']:.2f} | Gold: '{r['gold']}' | Pred: '{r['predicted']}'")

Среднее EM: 0.70
Среднее F1: 0.77

По вопросам:
  EM=1 F1=1.00 | Gold: 'yes' | Pred: 'Yes'
  EM=1 F1=1.00 | Gold: 'early 20s' | Pred: 'Early 20s'
  EM=1 F1=1.00 | Gold: 'the shootout' | Pred: 'Shootout'
  EM=0 F1=0.67 | Gold: 'Elfman' | Pred: 'Danny Elfman'
  EM=1 F1=1.00 | Gold: 'Comme des Garçons' | Pred: 'Comme des Garçons'
  EM=1 F1=1.00 | Gold: 'Teen Titans Go!' | Pred: 'Teen Titans Go!'
  EM=0 F1=0.00 | Gold: 'three' | Pred: 'John Glenn's, 3 times'
  EM=1 F1=1.00 | Gold: 'Brazil' | Pred: 'Brazil'
  EM=0 F1=0.00 | Gold: 'Steve Jobs' | Pred: 'Portrayal'
  EM=1 F1=1.00 | Gold: 'Pelle Almqvist' | Pred: 'Pelle Almqvist'


## Шаг 4. Метрика 2 — LLM-judge (оба судьи)

Судья получает вопрос, эталонный ответ и предсказанный ответ. Оценивает по шкале 0/1:
- **1** = предсказанный ответ фактически правильный (допускаются перефразировки, неполные формы).
- **0** = предсказанный ответ неверен или «I don't know».

Важно: судья не видит контекст — он оценивает только фактическую правильность ответа.
Это отличает его от RAGAS faithfulness (которая оценивает верность контексту).

Запускаем оба судьи на одних и тех же данных и потом сравниваем их согласие.

In [9]:
JUDGE_PROMPT = """You are an evaluation judge for a question answering system.

Question: {question}
Reference answer: {gold}
System answer: {predicted}

Is the system answer factually correct?
A system answer is correct if it conveys the same factual content as the reference answer,
even if phrased differently or slightly less complete.
"I don't know" or empty answers are always incorrect.

Respond with a JSON object:
{{"score": 0 or 1, "reason": "one sentence explanation"}}"""

def llm_judge(question, gold, predicted, model):
    prompt = JUDGE_PROMPT.format(
        question=question, gold=gold, predicted=predicted
    )
    raw = groq_complete(prompt, model=model, max_tokens=150, temperature=0.0)
    # Ищем JSON в ответе
    match = re.search(r'\{.*?\}', raw, re.DOTALL)
    if match:
        try:
            data = json.loads(match.group())
            score  = int(data.get("score", 0))
            reason = data.get("reason", "")
            return score, reason
        except (json.JSONDecodeError, ValueError):
            pass
    # Fallback: ищем 0 или 1 в ответе
    if '"score": 1' in raw or '"score":1' in raw:
        return 1, raw[:100]
    return 0, raw[:100]

print(f"Запускаю {JUDGE_MODEL_STRONG} как судью...")
for r in tqdm(results, desc="Judge 70B"):
    score, reason = llm_judge(r["question"], r["gold"], r["predicted"], JUDGE_MODEL_STRONG)
    r["judge_70b_score"]  = score
    r["judge_70b_reason"] = reason

print(f"\nЗапускаю {JUDGE_MODEL_FAST} как судью...")
for r in tqdm(results, desc="Judge 8B"):
    score, reason = llm_judge(r["question"], r["gold"], r["predicted"], JUDGE_MODEL_FAST)
    r["judge_8b_score"]  = score
    r["judge_8b_reason"] = reason

print(f"\nСредний judge_70b: {sum(r['judge_70b_score'] for r in results) / len(results):.2f}")
print(f"Средний judge_8b:  {sum(r['judge_8b_score']  for r in results) / len(results):.2f}")

Запускаю llama-3.3-70b-versatile как судью...


Judge 70B:   0%|          | 0/10 [00:00<?, ?it/s]


Запускаю llama-3.1-8b-instant как судью...


Judge 8B:   0%|          | 0/10 [00:00<?, ?it/s]


Средний judge_70b: 1.00
Средний judge_8b:  1.00


## Шаг 5. Метрика 3 — RAGAS

RAGAS измеряет две независимые оси:

- **Faithfulness** — насколько ответ подкреплён контекстом (0–1). Высокий = «ответ взят из документов». Низкий = «модель галлюцинировала».
- **Answer Relevance** — насколько ответ релевантен вопросу (0–1). Высокий = «отвечает на вопрос». Низкий = «говорит не о том».

**Важно:** RAGAS не знает, правильный ли ответ фактически — только верен ли он контексту и релевантен ли вопросу. Это другая ось, чем EM/F1/judge. Полезна для измерения галлюцинаций, а не точности.

In [10]:
FAITHFULNESS_PROMPT = """You are evaluating whether an answer is faithful to the given context.

Context:
{context}

Answer: {answer}

Task:
1. Break the answer into individual factual claims.
2. For each claim, find the exact sentence or phrase in the context that supports or contradicts it.
3. If no relevant text exists in the context — explicitly state that.

Return a JSON object:
{{
  "claims": [
    {{
      "claim": "the factual claim from the answer",
      "supported": true or false,
      "evidence": "the exact quote from the context that supports/contradicts this claim, or 'NOT FOUND IN CONTEXT' if absent"
    }}
  ],
  "faithfulness_score": <number of supported claims / total claims, from 0.0 to 1.0>
}}

Rules:
- evidence must be a direct quote from the context, not a paraphrase
- if the answer is "I don't know", return faithfulness_score 1.0 and empty claims list
- do not invent evidence that is not in the context"""

def compute_faithfulness(answer, context, model):
    context_short = context[:3000]
    prompt = FAITHFULNESS_PROMPT.format(context=context_short, answer=answer)
    raw = groq_complete(prompt, model=model, max_tokens=600, temperature=0.0)
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        try:
            data = json.loads(match.group())
            score  = float(data.get("faithfulness_score", float("nan")))
            claims = data.get("claims", [])
            return score, claims
        except (json.JSONDecodeError, ValueError):
            pass
    return float("nan"), []

print("Считаю faithfulness через LLM-judge (70B)...")
for r in tqdm(results):
    score, claims = compute_faithfulness(r["predicted"], r["context"], JUDGE_MODEL_STRONG)
    r["ragas_faithfulness"]     = score
    r["ragas_answer_relevancy"] = float("nan")
    r["faithfulness_claims"]    = claims

# Подробный вывод с доказательствами
print("\n" + "=" * 70)
print("FAITHFULNESS — ДЕТАЛЬНЫЙ РАЗБОР")
print("=" * 70)
for r in results:
    faith = r["ragas_faithfulness"]
    print(f"\nQ: {r['question'][:70]}")
    print(f"   Ответ:        '{r['predicted']}'")
    print(f"   Faithfulness: {faith:.2f}")
    for c in r["faithfulness_claims"]:
        icon = "✓" if c.get("supported") else "✗"
        print(f"   {icon} Утверждение: {c.get('claim', '')}")
        print(f"     Доказательство: {c.get('evidence', '')}")

print(f"\nСредняя faithfulness: {sum(r['ragas_faithfulness'] for r in results if not np.isnan(r['ragas_faithfulness'])) / len(results):.2f}")

Считаю faithfulness через LLM-judge (70B)...


  0%|          | 0/10 [00:00<?, ?it/s]


FAITHFULNESS — ДЕТАЛЬНЫЙ РАЗБОР

Q: Are both Jack and Coke and Clover Club Cocktail cocktails?
   Ответ:        'Yes'
   Faithfulness: 1.00

Q: What is the uppermost age range for the sort of fiction written by Ale
   Ответ:        'Early 20s'
   Faithfulness: 1.00
   ✓ Утверждение: The age range for young adult fiction includes the early 20s
     Доказательство: while authors and readers of 'young teen novels' often define it as written for those aged 15 to the early 20s.

Q: What other method of extending an ice hockey game exists other than th
   Ответ:        'Shootout'
   Faithfulness: 1.00
   ✓ Утверждение: The shootout is a method of extending an ice hockey game when the scores are tied after regulation.
     Доказательство: The two main methods of extending the game are the overtime period (commonly referred to as overtime) and the shootout.

Q: Between Danny Elfman and Fran Healy who has worked in more diverse fie
   Ответ:        'Danny Elfman'
   Faithfulness: 1.00
   ✓ Утв

## Шаг 6. Ручная разметка

Посмотрите на каждый вопрос и проставьте свою оценку: **1** = ответ правильный, **0** = неправильный.

Это ваш «золотой стандарт» для оценки метрик. Все автоматические метрики будут сравниваться с вашей оценкой.

In [11]:
print("=" * 70)
print("КАРТОЧКИ ДЛЯ РУЧНОЙ РАЗМЕТКИ")
print("=" * 70)
for i, r in enumerate(results):
    print(f"\n[{i+1}/10] Тип: {r['type']}")
    print(f"  Вопрос:    {r['question']}")
    print(f"  Эталон:    {r['gold']}")
    print(f"  Ответ RAG: {r['predicted']}")
    print(f"  EM={r['em']}  F1={r['f1']:.2f}  Judge70B={r['judge_70b_score']}  Judge8B={r['judge_8b_score']}")
    print(f"  RAGAS faith={r['ragas_faithfulness']:.2f}  rel={r['ragas_answer_relevancy']:.2f}")
    print(f"  70B reason: {r['judge_70b_reason']}")
    print(f"  8B  reason: {r['judge_8b_reason']}")

КАРТОЧКИ ДЛЯ РУЧНОЙ РАЗМЕТКИ

[1/10] Тип: comparison
  Вопрос:    Are both Jack and Coke and Clover Club Cocktail cocktails?
  Эталон:    yes
  Ответ RAG: Yes
  EM=1  F1=1.00  Judge70B=1  Judge8B=1
  RAGAS faith=1.00  rel=nan
  70B reason: The system answer 'Yes' conveys the same factual content as the reference answer 'yes', confirming that both Jack and Coke and Clover Club Cocktail are indeed cocktails.
  8B  reason: The system answer conveys the same factual content as the reference answer, even though it is phrased identically.

[2/10] Тип: bridge
  Вопрос:    What is the uppermost age range for the sort of fiction written by Alexander Gordon Smith?
  Эталон:    early 20s
  Ответ RAG: Early 20s
  EM=1  F1=1.00  Judge70B=1  Judge8B=1
  RAGAS faith=1.00  rel=nan
  70B reason: The system answer 'Early 20s' conveys the same factual content as the reference answer 'early 20s', differing only in capitalization.
  8B  reason: The system answer conveys the same factual content as the refe

In [14]:
# ================================================================
# ВРУЧНУЮ: 1 = правильный ответ, 0 = неправильный
# ================================================================

human_scores = [
    1,  # вопрос 1
    1,  # вопрос 2
    1,  # вопрос 3
    1,  # вопрос 4
    1,  # вопрос 5
    1,  # вопрос 6
    1,  # вопрос 7
    1,  # вопрос 8
    0,  # вопрос 9
    1,  # вопрос 10
]


print(f"Текущее состояние: {human_scores}")

Текущее состояние: [1, 1, 1, 1, 1, 1, 1, 1, 0, 1]


## Шаг 7. Сравнительный анализ метрик

Запускайте эту ячейку **после** заполнения `human_scores` выше.

In [15]:
assert all(s in (0, 1) for s in human_scores), "Заполните все human_scores значениями 0 или 1"
assert len(human_scores) == len(results)

for i, r in enumerate(results):
    r["human_score"] = human_scores[i]

print(f"Ручная оценка заполнена. Правильных: {sum(human_scores)}/{len(human_scores)}")

Ручная оценка заполнена. Правильных: 9/10


In [16]:
# Сводная таблица
rows = []
for i, r in enumerate(results):
    rows.append({
        "#":          i + 1,
        "Вопрос":     r["question"][:55] + "...",
        "Gold":       r["gold"][:30],
        "Predicted":  r["predicted"][:30],
        "Human":      r["human_score"],
        "EM":         r["em"],
        "F1":         round(r["f1"], 2),
        "Judge70B":   r["judge_70b_score"],
        "Judge8B":    r["judge_8b_score"],
        "Faith":      round(r["ragas_faithfulness"], 2),
        "Relevancy":  round(r["ragas_answer_relevancy"], 2),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

 #                                                     Вопрос              Gold             Predicted  Human  EM   F1  Judge70B  Judge8B  Faith  Relevancy
 1 Are both Jack and Coke and Clover Club Cocktail cocktai...               yes                   Yes      1   1 1.00         1        1    1.0        NaN
 2 What is the uppermost age range for the sort of fiction...         early 20s             Early 20s      1   1 1.00         1        1    1.0        NaN
 3 What other method of extending an ice hockey game exist...      the shootout              Shootout      1   1 1.00         1        1    1.0        NaN
 4 Between Danny Elfman and Fran Healy who has worked in m...            Elfman          Danny Elfman      1   0 0.67         1        1    1.0        NaN
 5 What Japanese fashion label was founded by the creator ... Comme des Garçons     Comme des Garçons      1   1 1.00         1        1    1.0        NaN
 6 Which of Tara Strong major voice role in animated serie...   Teen T

In [18]:
from sklearn.metrics import cohen_kappa_score

human  = [r["human_score"]     for r in results]
em     = [r["em"]              for r in results]
j_70b  = [r["judge_70b_score"] for r in results]
j_8b   = [r["judge_8b_score"]  for r in results]

# F1 бинаризуем по порогу 0.5 для сравнения с бинарными метриками
f1_bin = [1 if r["f1"] >= 0.5 else 0 for r in results]

# RAGAS faithfulness бинаризуем по 0.5
faith_bin = [1 if r["ragas_faithfulness"] >= 0.5 else 0 for r in results]

def agreement(a, b):
    """Процент совпадений и Cohen's Kappa."""
    pct = sum(x == y for x, y in zip(a, b)) / len(a)
    try:
        kappa = cohen_kappa_score(a, b)
    except Exception:
        kappa = float("nan")
    return pct, kappa

metrics = {
    "EM":                    em,
    "F1 (≥0.5)": f1_bin,
    "Judge 70B":             j_70b,
    "Judge 8B":              j_8b,
    "RAGAS faith (≥0.5)":    faith_bin,
}

print("=" * 60)
print("СОГЛАСИЕ С РУЧНОЙ ОЦЕНКОЙ")
print("=" * 60)
print(f"{'Метрика':<25} {'Совпадений':>12} {'Cohen Kappa':>12}")
print("-" * 50)
for name, scores in metrics.items():
    pct, kappa = agreement(human, scores)
    print(f"{name:<25} {pct:>11.0%} {kappa:>12.3f}")

print()
print("=" * 60)
print("СОГЛАСИЕ ДВУХ LLM-СУДЕЙ МЕЖДУ СОБОЙ")
print("=" * 60)
pct_jj, kappa_jj = agreement(j_70b, j_8b)
print(f"Judge 70B vs Judge 8B: {pct_jj:.0%} совпадений")


СОГЛАСИЕ С РУЧНОЙ ОЦЕНКОЙ
Метрика                     Совпадений  Cohen Kappa
--------------------------------------------------
EM                                80%        0.412
F1 (≥0.5)                         90%        0.615
Judge 70B                         90%        0.000
Judge 8B                          90%        0.000
RAGAS faith (≥0.5)                90%        0.000

СОГЛАСИЕ ДВУХ LLM-СУДЕЙ МЕЖДУ СОБОЙ
Judge 70B vs Judge 8B: 100% совпадений


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


In [19]:
# Детальный разбор расхождений между метриками
print("=" * 60)
print("РАСХОЖДЕНИЯ МЕТРИК С РУЧНОЙ ОЦЕНКОЙ")
print("=" * 60)

for name, scores in metrics.items():
    disagreements = [(i+1, human[i], scores[i]) for i in range(len(human)) if human[i] != scores[i]]
    if disagreements:
        print(f"\n{name} расходится с human на {len(disagreements)} вопросах:")
        for idx, h, m in disagreements:
            r = results[idx - 1]
            print(f"  [{idx}] Human={h}, Метрика={m}")
            print(f"       Q: {r['question'][:70]}")
            print(f"       Gold: '{r['gold']}' | Pred: '{r['predicted']}'")
    else:
        print(f"\n{name}: расхождений нет (100% согласие)")

РАСХОЖДЕНИЯ МЕТРИК С РУЧНОЙ ОЦЕНКОЙ

EM расходится с human на 2 вопросах:
  [4] Human=1, Метрика=0
       Q: Between Danny Elfman and Fran Healy who has worked in more diverse fie
       Gold: 'Elfman' | Pred: 'Danny Elfman'
  [7] Human=1, Метрика=0
       Q: This aircraft carrier served as a recovery ship for this flight which 
       Gold: 'three' | Pred: 'John Glenn's, 3 times'

F1 (≥0.5) расходится с human на 1 вопросах:
  [7] Human=1, Метрика=0
       Q: This aircraft carrier served as a recovery ship for this flight which 
       Gold: 'three' | Pred: 'John Glenn's, 3 times'

Judge 70B расходится с human на 1 вопросах:
  [9] Human=0, Метрика=1
       Q: Katherine Waterston and Chrisann Brennan has what in common?
       Gold: 'Steve Jobs' | Pred: 'Portrayal'

Judge 8B расходится с human на 1 вопросах:
  [9] Human=0, Метрика=1
       Q: Katherine Waterston and Chrisann Brennan has what in common?
       Gold: 'Steve Jobs' | Pred: 'Portrayal'

RAGAS faith (≥0.5) расходится с human 

In [20]:
# Расхождения между двумя судьями
print("=" * 60)
print("РАСХОЖДЕНИЯ МЕЖДУ ДВУМЯ LLM-СУДЬЯМИ")
print("=" * 60)

judge_disagreements = [i for i in range(len(results)) if results[i]["judge_70b_score"] != results[i]["judge_8b_score"]]
if judge_disagreements:
    for i in judge_disagreements:
        r = results[i]
        print(f"\n[{i+1}] Q: {r['question'][:70]}")
        print(f"     Gold: '{r['gold']}' | Pred: '{r['predicted']}'")
        print(f"     Human={r['human_score']} | 70B={r['judge_70b_score']} | 8B={r['judge_8b_score']}")
        print(f"     70B: {r['judge_70b_reason']}")
        print(f"     8B:  {r['judge_8b_reason']}")
else:
    print("Судьи согласны по всем вопросам.")

РАСХОЖДЕНИЯ МЕЖДУ ДВУМЯ LLM-СУДЬЯМИ
Судьи согласны по всем вопросам.


## Шаг 8. Рекомендация метрики и сохранение результатов

In [21]:
# Автоматическая рекомендация на основе Cohen's Kappa
kappas = {}
for name, scores in metrics.items():
    _, kappa = agreement(human, scores)
    kappas[name] = kappa

best_metric = max(kappas, key=kappas.get)

print("=" * 60)
print("ИТОГИ И РЕКОМЕНДАЦИЯ")
print("=" * 60)
print(f"\nРанжирование метрик по Cohen's Kappa (согласие с человеком):")
for name, k in sorted(kappas.items(), key=lambda x: -x[1]):
    bar = "█" * int(max(0, k) * 20)
    print(f"  {name:<25} κ={k:.3f} {bar}")

print(f"\nЛучшая метрика по κ: {best_metric}")
print(f"Согласие двух судей: Kappa = {kappa_jj:.3f}")

print("""
Рекомендации для выбора метрики в ноутбуке 03:

  Judge 70B  — если нужна максимальная точность оценки, готовы тратить токены.
  Judge 8B   — если нужна скорость и экономия (приемлемо если Kappa с 70B > 0.7).
  F1         — если нужна воспроизводимость без LLM (всегда детерминирован).
  EM         — слишком строгий для HotpotQA (много False Negative на парафразах).
  RAGAS      — оставляем как дополнительную метрику для faithfulness / галлюцинаций.

Оптимальная стратегия для диплома: основная метрика = Judge 70B (если Kappa с human > 0.7),
дополнительно F1 (для сравнимости с литературой), RAGAS faithfulness (для анализа галлюцинаций).
""")

# Сохраняем всё
eval_results = {
    "eval_questions": N_EVAL,
    "generator_model": GENERATOR_MODEL,
    "judge_strong": JUDGE_MODEL_STRONG,
    "judge_fast": JUDGE_MODEL_FAST,
    "kappas_vs_human": kappas,
    "inter_judge_kappa": kappa_jj,
    "best_metric_by_kappa": best_metric,
    "results": results,
}

with open(EVAL_DIR / "metrics_validation.json", "w", encoding="utf-8") as f:
    json.dump(eval_results, f, ensure_ascii=False, indent=2)

df.to_csv(EVAL_DIR / "metrics_table.csv", index=False)

print(f"Результаты сохранены в {EVAL_DIR}")

ИТОГИ И РЕКОМЕНДАЦИЯ

Ранжирование метрик по Cohen's Kappa (согласие с человеком):
  F1 (≥0.5)                 κ=0.615 ████████████
  EM                        κ=0.412 ████████
  Judge 70B                 κ=0.000 
  Judge 8B                  κ=0.000 
  RAGAS faith (≥0.5)        κ=0.000 

Лучшая метрика по κ: F1 (≥0.5)
Согласие двух судей: Kappa = nan

Рекомендации для выбора метрики в ноутбуке 03:

  Judge 70B  — если нужна максимальная точность оценки, готовы тратить токены.
  Judge 8B   — если нужна скорость и экономия (приемлемо если Kappa с 70B > 0.7).
  F1         — если нужна воспроизводимость без LLM (всегда детерминирован).
  EM         — слишком строгий для HotpotQA (много False Negative на парафразах).
  RAGAS      — оставляем как дополнительную метрику для faithfulness / галлюцинаций.

Оптимальная стратегия для диплома: основная метрика = Judge 70B (если Kappa с human > 0.7),
дополнительно F1 (для сравнимости с литературой), RAGAS faithfulness (для анализа галлюцинаций).

Ре